In [0]:
!pip install lightgbm shap

In [0]:
# ============================================
# CROSS-CITY SHAP DEPENDENCY DATASET (LIGHTGBM)
# Shared feature: hour
# Output: cross_city_dependency_hour.csv
# ============================================

import os
import gc
import numpy as np
import pandas as pd
import lightgbm as lgb
import shap

# -----------------------
# CONFIG
# -----------------------
PROJECT_FOLDER = "DAMO_699-4-Capstone-Project"
OUTPUT_ROOT = "output"
FEATURE_NAME = "hour"
LABEL_COL = "delay_indicator"
SAMPLE_SIZE = 200000
SEED = 42

CITY_CONFIG = {
    "Toronto": "workspace.capstone_project.toronto_model_ready",
    "NYC": "workspace.capstone_project.nyc_model_ready"
}

CATEGORICAL_COLS = ["incident_category", "season", "unified_call_source", "location_area"]
NUMERIC_COLS = [
    "hour", "day_of_week", "month", "year",
    "unified_alarm_level", "calls_past_30min", "calls_past_60min"
]

# -----------------------
# PATH SETUP
# -----------------------
cwd = os.getcwd()
print("Current working directory:", cwd)

if PROJECT_FOLDER not in cwd:
    raise ValueError(
        f"Run this notebook from inside '{PROJECT_FOLDER}'. Current dir = {cwd}"
    )

project_root = cwd[:cwd.index(PROJECT_FOLDER) + len(PROJECT_FOLDER)]
out_dir = os.path.join(project_root, OUTPUT_ROOT, "shap", "cross_city_comparison")
os.makedirs(out_dir, exist_ok=True)

print("Saving outputs to:", out_dir)

# -----------------------
# HELPERS
# -----------------------
def load_sample_from_spark(table_name: str, city_name: str, sample_size: int) -> pd.DataFrame:
    print(f"\nLoading {city_name} from {table_name} ...")
    
    sdf = spark.table(table_name).filter(f"{LABEL_COL} IS NOT NULL")
    
    keep_cols = [c for c in (CATEGORICAL_COLS + NUMERIC_COLS + [LABEL_COL]) if c in sdf.columns]
    sdf = sdf.select(*keep_cols)
    
    print(f"{city_name} label distribution:")
    sdf.groupBy(LABEL_COL).count().orderBy(LABEL_COL).show()
    
    # Sample for SHAP-friendly local modeling
    pdf = sdf.sample(withReplacement=False, fraction=1.0, seed=SEED).limit(sample_size).toPandas()
    print(f"{city_name} sampled rows:", len(pdf))
    return pdf


def prepare_features(pdf: pd.DataFrame):
    pdf = pdf.copy()
    
    # Keep only required columns actually present
    feature_cols = [c for c in (CATEGORICAL_COLS + NUMERIC_COLS) if c in pdf.columns]
    X = pdf[feature_cols].copy()
    y = pdf[LABEL_COL].astype(int).copy()
    
    # Cast categoricals for LightGBM
    cat_cols_present = [c for c in CATEGORICAL_COLS if c in X.columns]
    for c in cat_cols_present:
        X[c] = X[c].astype("category")
    
    return X, y, cat_cols_present


def train_lightgbm(X: pd.DataFrame, y: pd.Series, cat_cols_present: list, city_name: str):
    print(f"Training LightGBM for {city_name} ...")
    
    # Handle class imbalance
    pos = int(y.sum())
    neg = int((y == 0).sum())
    scale_pos_weight = neg / max(pos, 1)
    
    model = lgb.LGBMClassifier(
        objective="binary",
        n_estimators=200,
        learning_rate=0.05,
        num_leaves=31,
        max_depth=-1,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=SEED,
        scale_pos_weight=scale_pos_weight
    )
    
    model.fit(X, y, categorical_feature=cat_cols_present)
    print(f"{city_name} LightGBM trained.")
    return model


def build_dependency_df(X: pd.DataFrame, model, city_name: str, feature_name: str) -> pd.DataFrame:
    print(f"Computing SHAP dependency values for {city_name} ...")
    
    explainer = shap.TreeExplainer(model)
    shap_values = explainer.shap_values(X)
    
    # Binary-class handling
    if isinstance(shap_values, list):
        shap_matrix = shap_values[1]
    else:
        shap_matrix = shap_values
    
    feature_index = X.columns.get_loc(feature_name)
    
    dep_df = pd.DataFrame({
        "city": city_name,
        "feature": feature_name,
        "feature_value": X[feature_name].astype(float).values,
        "shap_value": shap_matrix[:, feature_index]
    })
    
    print(f"{city_name} dependency rows:", len(dep_df))
    return dep_df


# -----------------------
# MAIN
# -----------------------
all_dep = []

for city_name, table_name in CITY_CONFIG.items():
    pdf = load_sample_from_spark(table_name, city_name, SAMPLE_SIZE)
    X, y, cat_cols_present = prepare_features(pdf)
    
    if FEATURE_NAME not in X.columns:
        raise ValueError(f"Feature '{FEATURE_NAME}' not found in {city_name} dataset.")
    
    model = train_lightgbm(X, y, cat_cols_present, city_name)
    dep_df = build_dependency_df(X, model, city_name, FEATURE_NAME)
    all_dep.append(dep_df)
    
    del pdf, X, y, model, dep_df
    gc.collect()

# Combine both cities
cross_city_dep = pd.concat(all_dep, ignore_index=True)

# Save CSV
out_csv = os.path.join(out_dir, "cross_city_dependency_hour.csv")
cross_city_dep.to_csv(out_csv, index=False)

print("\nSaved dependency CSV to:")
print(out_csv)

print("\nPreview:")
print(cross_city_dep.head())
print("\nShape:", cross_city_dep.shape)